# Titanic Survival Prediction Workflow

This notebook implements **Phase 1 to Phase 5** for predicting Titanic survival.

## Goals
- Predict **individual passenger survival** (`0` or `1`)
- Estimate the **overall predicted survival rate (%)**
- Compare multiple models and select the best one using clear metrics

## Phases
1. Setup and data intake
2. Preprocessing and feature preparation
3. Model training and selection
4. Prediction outputs
5. Reporting and reproducibility

## Phase 1: Setup and Data Intake

In this phase, we:
- import required libraries
- load the Titanic dataset
- validate schema and target
- run quick integrity checks

In [11]:
# Phase 1 - Imports and basic configuration
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

# Make notebook outputs easier to read
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

RANDOM_STATE = 42

In [12]:
# Phase 1 - Load dataset and validate structure
DATA_PATH = Path('Titanic-Dataset.csv')

df = pd.read_csv(DATA_PATH)

print(f'Dataset path: {DATA_PATH.resolve()}')
print(f'Shape: {df.shape}')
print('\nColumns:')
print(df.columns.tolist())

# Confirm expected target exists
if 'Survived' not in df.columns:
    raise ValueError("Target column 'Survived' not found in dataset.")

print('\nDtypes:')
print(df.dtypes)

print('\nFirst 5 rows:')
display(df.head())

Dataset path: /home/eiamin/AI-Engineering-Course/Module-7/Titanic/Titanic-Dataset.csv
Shape: (891, 12)

Columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Dtypes:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

First 5 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
# Phase 1 - Integrity checks and quick data quality profile
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print('Missing value summary (top):')
display(missing_summary.head(12))

duplicate_rows = df.duplicated().sum()
print(f'Duplicate rows: {duplicate_rows}')

if 'PassengerId' in df.columns:
    unique_passenger_ids = df['PassengerId'].nunique()
    print(f"Unique PassengerId: {unique_passenger_ids} / {len(df)}")

# Basic invalid value checks for numeric fields
if 'Age' in df.columns:
    invalid_age = (df['Age'] < 0).sum(skipna=True)
    print(f'Invalid Age (<0): {invalid_age}')

if 'Fare' in df.columns:
    invalid_fare = (df['Fare'] < 0).sum(skipna=True)
    print(f'Invalid Fare (<0): {invalid_fare}')

print('\nTarget distribution (Survived):')
print(df['Survived'].value_counts(dropna=False))
print((df['Survived'].value_counts(normalize=True) * 100).round(2).rename('percent'))

Missing value summary (top):


,missing_count,missing_pct
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22
PassengerId,0,0.00
Name,0,0.00
Pclass,0,0.00
Survived,0,0.00
Sex,0,0.00
Parch,0,0.00
SibSp,0,0.00


Duplicate rows: 0
Unique PassengerId: 891 / 891
Invalid Age (<0): 0
Invalid Fare (<0): 0

Target distribution (Survived):
Survived
0    549
1    342
Name: count, dtype: int64
Survived
0    61.62
1    38.38
Name: percent, dtype: float64


## Phase 2: Preprocessing and Feature Preparation

In this phase, we:
- create engineered features (`FamilySize`, `IsAlone`, `CabinKnown`, `Title`)
- define feature columns
- split train/test data
- build reusable preprocessing pipelines for numeric and categorical data

In [14]:
# Phase 2 - Feature engineering
work_df = df.copy()

# Family-based features
work_df['FamilySize'] = work_df['SibSp'] + work_df['Parch'] + 1
work_df['IsAlone'] = (work_df['FamilySize'] == 1).astype(int)

# Cabin availability indicator (instead of dropping all cabin information)
work_df['CabinKnown'] = work_df['Cabin'].notna().astype(int)

# Extract title from passenger name (useful social-status signal)
work_df['Title'] = work_df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
work_df['Title'] = work_df['Title'].fillna('Unknown').str.strip()

# Keep rare titles grouped to reduce noise
title_counts = work_df['Title'].value_counts()
rare_titles = title_counts[title_counts < 10].index
work_df['Title'] = work_df['Title'].replace(rare_titles, 'Rare')

# Define target and features
y = work_df['Survived'].astype(int)

feature_cols = [
    'Pclass', 'Sex', 'Age', 'Fare', 'Embarked',
    'FamilySize', 'IsAlone', 'CabinKnown', 'Title'
]

X = work_df[feature_cols]

print('Feature columns used for modeling:')
print(feature_cols)
print(f'X shape: {X.shape}, y shape: {y.shape}')

Feature columns used for modeling:
['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone', 'CabinKnown', 'Title']
X shape: (891, 9), y shape: (891,)


In [15]:
# Phase 2 - Train/test split and preprocessing pipelines
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

numeric_features = ['Age', 'Fare', 'FamilySize']
# Keep IsAlone, CabinKnown, Pclass as non-scaled numeric-like categorical indicators
categorical_features = ['Sex', 'Embarked', 'Title', 'Pclass', 'IsAlone', 'CabinKnown']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print('Preprocessing pipeline is ready.')

Train shape: (712, 9), Test shape: (179, 9)
Preprocessing pipeline is ready.


## Phase 3: Model Training and Selection

In this phase, we:
- train 3 candidate models using the same preprocessing
- compare model metrics (Accuracy, Precision, Recall, F1, ROC-AUC)
- select the best model using: **F1 first, ROC-AUC second, Accuracy third**

In [16]:
# Phase 3 - Define models and evaluate on holdout test set
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
    ),
    'KNN': KNeighborsClassifier(n_neighbors=11),
}

results = []
trained_pipelines = {}

for model_name, model in models.items():
    # Combine preprocessing and model into one pipeline
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model),
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # Try probability-based ROC-AUC when available
    if hasattr(clf, 'predict_proba'):
        y_proba = clf.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        y_proba = None
        roc_auc = np.nan

    row = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC': roc_auc,
    }
    results.append(row)
    trained_pipelines[model_name] = clf

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['F1', 'ROC_AUC', 'Accuracy'], ascending=False).reset_index(drop=True)

print('Model comparison table:')
display(results_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'ROC_AUC': '{:.4f}',
}))

Model comparison table:


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,LogisticRegression,0.8268,0.7879,0.7536,0.7704,0.8713
1,KNN,0.8156,0.8000,0.6957,0.7442,0.8665
2,RandomForest,0.7933,0.7667,0.6667,0.7132,0.8478


In [17]:
# Phase 3 - Select best model and inspect detailed performance
best_model_name = results_df.loc[0, 'Model']
best_pipeline = trained_pipelines[best_model_name]

print(f'Best model selected: {best_model_name}')

best_y_pred = best_pipeline.predict(X_test)
print('\nClassification Report (Best Model):')
print(classification_report(y_test, best_y_pred, digits=4))

print('Confusion Matrix (Best Model):')
print(confusion_matrix(y_test, best_y_pred))

Best model selected: LogisticRegression

Classification Report (Best Model):
              precision    recall  f1-score   support

           0     0.8496    0.8727    0.8610       110
           1     0.7879    0.7536    0.7704        69

    accuracy                         0.8268       179
   macro avg     0.8187    0.8132    0.8157       179
weighted avg     0.8258    0.8268    0.8261       179

Confusion Matrix (Best Model):
[[96 14]
 [17 52]]


## Phase 4: Prediction Outputs

In this phase, we:
- retrain the selected best model on the full dataset
- generate individual passenger predictions
- compute the overall predicted survival rate

In [18]:
# Phase 4 - Retrain best model on full data and create predictions
# Rebuild the final pipeline from the winning estimator type for clean reproducibility
best_estimator_template = models[best_model_name]

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', best_estimator_template),
])

# Fit on full data since the goal is generating predictions for all passengers
final_pipeline.fit(X, y)

# Individual survival class predictions
predicted_survived = final_pipeline.predict(X)

# Predicted probability of survival (class 1), when supported
if hasattr(final_pipeline, 'predict_proba'):
    predicted_probability = final_pipeline.predict_proba(X)[:, 1]
else:
    predicted_probability = np.full(shape=len(X), fill_value=np.nan)

prediction_df = pd.DataFrame({
    'PassengerId': work_df['PassengerId'],
    'Name': work_df['Name'],
    'ActualSurvived': y,
    'PredictedSurvived': predicted_survived.astype(int),
    'PredictedSurvivalProbability': predicted_probability,
})

overall_predicted_survival_rate = prediction_df['PredictedSurvived'].mean() * 100

print(f'Overall predicted survival rate: {overall_predicted_survival_rate:.2f}%')
print('\nSample predictions:')
display(prediction_df.head(10))

Overall predicted survival rate: 36.36%

Sample predictions:


,PassengerId,Name,ActualSurvived,PredictedSurvived,PredictedSurvivalProbability
0,1,"Braund, Mr. Owen Harris",0,0,0.077038
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,1,0.968422
2,3,"Heikkinen, Miss. Laina",1,1,0.608469
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,1,0.953795
4,5,"Allen, Mr. William Henry",0,0,0.066339
5,6,"Moran, Mr. James",0,0,0.105248
6,7,"McCarthy, Mr. Timothy J",0,0,0.305624
7,8,"Palsson, Master. Gosta Leonard",0,0,0.392004
8,9,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",1,1,0.607172
9,10,"Nasser, Mrs. Nicholas (Adele Achem)",1,1,0.938133


## Phase 5: Reporting and Reproducibility

In this phase, we:
- summarize selected model and key metrics
- provide reproducible outputs
- export prediction results for reuse

In [19]:
# Phase 5 - Final report objects
best_row = results_df.iloc[0].to_dict()

final_report = {
    'BestModel': best_model_name,
    'BestModelMetrics': {
        'Accuracy': round(float(best_row['Accuracy']), 4),
        'Precision': round(float(best_row['Precision']), 4),
        'Recall': round(float(best_row['Recall']), 4),
        'F1': round(float(best_row['F1']), 4),
        'ROC_AUC': round(float(best_row['ROC_AUC']), 4) if pd.notna(best_row['ROC_AUC']) else np.nan,
    },
    'DatasetRows': int(len(work_df)),
    'OverallPredictedSurvivalRatePercent': round(float(overall_predicted_survival_rate), 2),
    'FeatureColumnsUsed': feature_cols,
    'RandomState': RANDOM_STATE,
}

print('Final prediction workflow report:')
for k, v in final_report.items():
    print(f'{k}: {v}')

Final prediction workflow report:
BestModel: LogisticRegression
BestModelMetrics: {'Accuracy': 0.8268, 'Precision': 0.7879, 'Recall': 0.7536, 'F1': 0.7704, 'ROC_AUC': 0.8713}
DatasetRows: 891
OverallPredictedSurvivalRatePercent: 36.36
FeatureColumnsUsed: ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone', 'CabinKnown', 'Title']
RandomState: 42


In [20]:
# Phase 5 - Export predictions to CSV (optional but recommended)
OUTPUT_PATH = Path('titanic_survival_predictions.csv')
prediction_df.to_csv(OUTPUT_PATH, index=False)

print(f'Prediction file saved to: {OUTPUT_PATH.resolve()}')
print('Use this file for downstream analysis or dashboarding.')

Prediction file saved to: /home/eiamin/AI-Engineering-Course/Module-7/Titanic/titanic_survival_predictions.csv
Use this file for downstream analysis or dashboarding.


## Run Order

Execute cells from top to bottom:
1. Phase 1 cells
2. Phase 2 cells
3. Phase 3 cells
4. Phase 4 cells
5. Phase 5 cells

This ensures all required variables and pipelines are available in sequence.